In [1]:
from langchain_community.document_loaders import TextLoader
import os

documents = []

for file in os.listdir("documents"):
    loader = TextLoader(f"documents/{file}")
    documents.extend(loader.load())

print(documents[:2])

/home/shivansh/Desktop/OpenSource/interneers-lab/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'documents/product_manual.txt'}, page_content='Lego Castle Product Manual\n\nThe Lego Castle construction set by LEGO includes over 500 interlocking pieces used to build a medieval castle with towers, gates, and knight figures.\n\nRecommended age: 8 years and above.\n\nWarranty:\nThe Lego Castle comes with a 1 year manufacturer warranty covering missing or damaged pieces.\n\nAssembly:\nAssembly instructions are included inside the package. Adult supervision is recommended for younger children.\n\nSafety:\nContains small parts. Not suitable for children under 3 years of age.\n\n\nLego City Set Product Manual\n\nThe Lego City Set allows children to construct roads, buildings, and emergency vehicles using modular construction bricks.\n\nRecommended age: 7 years and above.\n\nWarranty:\nIncludes a 1 year manufacturer warranty.\n\nMaintenance:\nStore pieces in a dry environment to avoid discoloration.\n\n\nWooden Blocks Product Manual\n\nWooden Blocks by FunWoo

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks[0].page_content)

Lego Castle Product Manual

The Lego Castle construction set by LEGO includes over 500 interlocking pieces used to build a medieval castle with towers, gates, and knight figures.

Recommended age: 8 years and above.


In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_93770/384917042.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/home/shivansh/Desktop/OpenSource/interneers-lab/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [5]:
def retrieve_relevant_chunks(query, top_k=3):
    docs = vectorstore.similarity_search(query, k=top_k)

    return [doc.page_content for doc in docs]

In [6]:
query = "What's the return policy for damaged items?"

results = retrieve_relevant_chunks(query)

for r in results:
    print(r)
    print("=" * 50)

Toy Store Return Policy

Customers may return damaged or defective products within 30 days of delivery.

Refunds are processed within 5 business days after successful inspection.
Toy Store Return Policy

Customers may return damaged or defective products within 30 days of delivery.

Refunds are processed within 5 business days after successful inspection.
Conditions for return:
- Product must include original packaging.
- Missing parts should be reported within 7 days.
- Products damaged due to misuse are not eligible for refund.

Replacement Policy:
Construction toys with missing pieces may qualify for replacement parts instead of a refund.


In [7]:
RETRIEVAL_TESTS = [
    {
        "query": "damaged item return",
        "expected_keyword": "30 days"
    },
    {
        "query": "lego warranty",
        "expected_keyword": "1 year"
    }
]

In [8]:
def evaluate_retrieval():
    passed = 0

    for test in RETRIEVAL_TESTS:
        results = retrieve_relevant_chunks(test["query"])

        combined = " ".join(results).lower()

        success = test["expected_keyword"].lower() in combined

        print(f"\nQuery: {test['query']}")
        print("PASS" if success else "FAIL")

        if success:
            passed += 1

    print(f"\nScore: {passed}/{len(RETRIEVAL_TESTS)}")

In [9]:
!pip install google-genai

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [10]:
from google import genai
import os

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [11]:
def ask_expert(question):
    retrieved_docs = retrieve_relevant_chunks(question)

    context = "\n\n".join(retrieved_docs)

    prompt = f"""
    Answer the question using ONLY the context below.

    Context:
    {context}

    Question:
    {question}
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [12]:
ask_expert("What is the warranty period for the Lego Castle?")

'The Lego Castle comes with a 1 year manufacturer warranty.'

In [13]:
RAG_TESTS = [
    {
        "question": "What is the warranty period for the Lego Castle?",
        "expected": "1 year"
    },
    {
        "question": "How long can damaged items be returned?",
        "expected": "30 days"
    },
    {
        "question": "Which toys are educational?",
        "expected": "Wooden Blocks"
    }
]

In [14]:
def evaluate_rag():
    passed = 0

    for test in RAG_TESTS:
        answer = ask_expert(test["question"])

        success = test["expected"].lower() in answer.lower()

        print(f"\nQuestion: {test['question']}")
        print("Answer:", answer)
        print("PASS" if success else "FAIL")

        if success:
            passed += 1

    print(f"\nRAG Score: {passed}/{len(RAG_TESTS)}")

In [15]:
evaluate_retrieval()


Query: damaged item return
PASS

Query: lego warranty
PASS

Score: 2/2


In [16]:
evaluate_rag()


Question: What is the warranty period for the Lego Castle?
Answer: The warranty period for the Lego Castle is 1 year.
PASS

Question: How long can damaged items be returned?
Answer: Damaged items can be returned within 30 days of delivery.
PASS

Question: Which toys are educational?
Answer: Wooden Blocks and Soft Building Blocks are educational toys.
PASS

RAG Score: 3/3
